# Manage rule sets

List and delete SHACL rule sets (and their versions) in the CDF project, plus the data products that reference them.

Rule sets cannot be updated in place, and CDF refuses to delete a rule set that still has versions. When you change a rule set (for example its `name`), `cdf deploy` fails with `Rule set '...' can not be deleted, because it contains versions`. Delete the rule set here, then run `cdf deploy` again.

CDF also refuses to delete a rule set version that a data product version references. `delete_rulesets` therefore deletes those data product versions first.

Credentials come from the Toolkit `.env` next to `cdf.toml` (`CDF_*` / `IDP_*`), same as `deploy_infrastructure.py`.

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

SCRIPTS_DIR = next(
    path for path in (Path.cwd(), Path.cwd() / "modules/common/cdf_dq_runtime/scripts") if (path / "_cli.py").is_file()
)
sys.path.insert(0, str(SCRIPTS_DIR))

from _cli import configure_deploy_cli, create_client_from_env, load_project_env  # noqa: E402

configure_deploy_cli()
print(f"Credentials: {load_project_env()}")
client = create_client_from_env()
API_URL = f"/api/v1/projects/{client.config.project}"
RULESETS_URL = f"{API_URL}/rulesets"
DATA_PRODUCTS_URL = f"{API_URL}/dataproducts"
ALPHA_HEADERS = {"cdf-version": "alpha"}
print(f"Project: {client.config.project}")

In [ ]:
def _list_items(url: str, limit: int = 100) -> list[dict]:
    items: list[dict] = []
    params: dict[str, object] = {"limit": limit}
    while True:
        body = client.get(url, params=params, headers=ALPHA_HEADERS).json()
        items.extend(body.get("items", []))
        if not body.get("nextCursor"):
            return items
        params["cursor"] = body["nextCursor"]


def _post(url: str, items: list[dict]) -> None:
    client.post(url, json={"items": items}, headers=ALPHA_HEADERS)


def _prefix(dry_run: bool) -> str:
    return "[DRY RUN] Would delete" if dry_run else "Deleted"


def list_ruleset_versions(external_id: str) -> list[str]:
    """Return the versions of a rule set."""
    return [str(item["version"]) for item in _list_items(f"{RULESETS_URL}/{external_id}/versions")]


def list_rulesets() -> list[dict]:
    """Print and return all rule sets in the project with their versions."""
    rulesets = _list_items(RULESETS_URL)
    for ruleset in rulesets:
        ruleset["versions"] = list_ruleset_versions(ruleset["externalId"])
        print(f"{ruleset['externalId']} | {ruleset.get('name')} | versions: {ruleset['versions']}")
    print(f"{len(rulesets)} rule set(s)")
    return rulesets


def list_data_product_versions(external_id: str) -> list[dict]:
    """Return the versions of a data product, including their `quality.rules` references."""
    return _list_items(f"{DATA_PRODUCTS_URL}/{external_id}/versions", limit=10)


def list_data_products() -> list[dict]:
    """Print and return all data products with their versions and referenced rule set versions."""
    data_products = _list_items(DATA_PRODUCTS_URL)
    for data_product in data_products:
        data_product["versions"] = list_data_product_versions(data_product["externalId"])
        print(f"{data_product['externalId']} | {data_product.get('name')}")
        for version in data_product["versions"]:
            rules = [f"{rule['externalId']}:{rule['version']}" for rule in version.get("quality", {}).get("rules", [])]
            print(f"    {version['version']} ({version.get('status')}) | rules: {rules}")
    print(f"{len(data_products)} data product(s)")
    return data_products


def data_product_versions_referencing(ruleset_ids: list[str]) -> list[tuple[str, str]]:
    """Return (data product, version) pairs whose `quality.rules` reference any of the rule sets."""
    referencing: list[tuple[str, str]] = []
    for data_product in _list_items(DATA_PRODUCTS_URL):
        for version in list_data_product_versions(data_product["externalId"]):
            rules = version.get("quality", {}).get("rules", [])
            if any(rule["externalId"] in ruleset_ids for rule in rules):
                referencing.append((data_product["externalId"], version["version"]))
    return referencing


def delete_data_product_versions(ids: list[tuple[str, str]], dry_run: bool = True) -> None:
    """Delete data product versions given as (data product, version) pairs."""
    for external_id, version in ids:
        if not dry_run:
            _post(f"{DATA_PRODUCTS_URL}/{external_id}/versions/delete", [{"version": version}])
        print(f"{_prefix(dry_run)} data product version {external_id}:{version}")


def delete_data_products(external_ids: list[str], dry_run: bool = True) -> None:
    """Delete data products, removing all their versions first, and confirm they are gone."""
    if not external_ids:
        print("No data products selected.")
        return
    for external_id in external_ids:
        versions = [(external_id, version["version"]) for version in list_data_product_versions(external_id)]
        delete_data_product_versions(versions, dry_run)
        if not dry_run:
            _post(f"{DATA_PRODUCTS_URL}/delete", [{"externalId": external_id}])
        print(f"{_prefix(dry_run)} data product {external_id}")
    if dry_run:
        return
    remaining = {data_product["externalId"] for data_product in _list_items(DATA_PRODUCTS_URL)}
    for external_id in external_ids:
        status = "still exists in CDF" if external_id in remaining else "confirmed deleted from CDF"
        print(f"Data product {external_id}: {status}")


def delete_rulesets(external_ids: list[str], dry_run: bool = True) -> None:
    """Delete rule sets with all their versions, and the data product versions that reference them.

    Args:
        external_ids: Rule sets to delete.
        dry_run: Only print what would be deleted.
    """
    delete_data_product_versions(data_product_versions_referencing(external_ids), dry_run)
    for external_id in external_ids:
        for version in list_ruleset_versions(external_id):
            if not dry_run:
                _post(f"{RULESETS_URL}/{external_id}/versions/delete", [{"version": version}])
            print(f"{_prefix(dry_run)} rule set version {external_id}:{version}")
        if not dry_run:
            _post(f"{RULESETS_URL}/delete", [{"externalId": external_id}])
        print(f"{_prefix(dry_run)} rule set {external_id}")

## Select and delete rule sets and data products

The cell lists the rule sets and data products in CDF and shows them as selection lists (Ctrl/Shift-click to select several).

- **Rule sets:** deletes all versions and the rule set. Data product versions that reference them are deleted first.
- **Data products:** deletes all versions and the data product, for example one no longer deployed by this module.

Keep **Dry run** checked to review what would be deleted, then uncheck it and click **Delete selected** again. After a real delete the lists refresh from CDF. Run `cdf deploy` afterwards to recreate what this module defines.

In [ ]:
layout = widgets.Layout(width="95%", height="160px")
style = {"description_width": "110px"}
ruleset_select = widgets.SelectMultiple(description="Rule sets", layout=layout, style=style)
data_product_select = widgets.SelectMultiple(description="Data products", layout=layout, style=style)
dry_run_box = widgets.Checkbox(value=True, description="Dry run (only show what would be deleted)", style=style)
delete_button = widgets.Button(description="Delete selected", button_style="danger", icon="trash")
output = widgets.Output()


def refresh() -> None:
    """List rule sets and data products in CDF and offer them for selection."""
    rulesets = list_rulesets()
    print()
    data_products = list_data_products()
    ruleset_select.options = [
        (f"{ruleset['externalId']} | {ruleset.get('name')} | versions: {ruleset['versions']}", ruleset["externalId"])
        for ruleset in rulesets
    ]
    data_product_select.options = [
        (
            f"{data_product['externalId']} | {data_product.get('name')} | "
            f"versions: {[version['version'] for version in data_product['versions']]}",
            data_product["externalId"],
        )
        for data_product in data_products
    ]


def on_delete(_: widgets.Button) -> None:
    output.clear_output()
    with output:
        rulesets_to_delete = list(ruleset_select.value)
        data_products_to_delete = list(data_product_select.value)
        if not rulesets_to_delete and not data_products_to_delete:
            print("Nothing selected.")
            return
        if rulesets_to_delete:
            delete_rulesets(rulesets_to_delete, dry_run=dry_run_box.value)
        if data_products_to_delete:
            delete_data_products(data_products_to_delete, dry_run=dry_run_box.value)
        if not dry_run_box.value:
            print("\nRemaining in CDF:")
            refresh()


delete_button.on_click(on_delete)
display(widgets.VBox([ruleset_select, data_product_select, dry_run_box, delete_button, output]))
with output:
    refresh()